# Sesion 6 - Flujo multiagente con modelo local

En este notebook ejecutamos un sistema multiagente sencillo usando el modelo local que venimos usando en el curso: `qwen2.5:3b` via Ollama.

La arquitectura tiene cinco llamadas al modelo:

1. `Planner`: define el plan de trabajo.
2. `Researcher`: convierte evidencia recuperada en hallazgos.
3. `Writer`: redacta un borrador ejecutivo.
4. `Reviewer`: revisa soporte, riesgos y limitaciones.
5. `Final Writer`: ajusta la respuesta final con base en la revision.

## Requisitos

Antes de ejecutar el notebook, Ollama debe estar corriendo y el modelo debe estar descargado:

```bash
ollama pull qwen2.5:3b
```

Este ejemplo no requiere CrewAI ni AutoGen. La idea es entender primero la arquitectura base.

In [ ]:
from agents.multiagent_collaboration import ClassroomMultiAgentSystem


MODEL_NAME = "qwen2.5:3b"

question = (
    "Debe NovaRetail lanzar descuentos agresivos en ciudades intermedias "
    "durante Q4 para recuperar ventas?"
)

question

## Crear el sistema multiagente

El sistema usa los documentos Markdown de `data/rag_business_case` como evidencia del caso NovaRetail.

La recuperacion de evidencia usa una busqueda por palabras clave intencionalmente simple para que el flujo sea transparente. En una version mas avanzada, este paso podria reemplazarse por el RAG construido en sesiones 4 y 5.

In [ ]:
multiagent_system = ClassroomMultiAgentSystem(
    model_name=MODEL_NAME,
    top_k=5,
)

evidence = multiagent_system.retrieve_evidence(question)

for item in evidence:
    print(f"{item.chunk_id} | score={item.score} | {item.section} | {item.source}")

## Ejecutar el flujo completo

Esta celda hace cinco llamadas al modelo local. Puede tardar unos segundos dependiendo del equipo.

In [ ]:
result = multiagent_system.answer(question)

print(result.final_answer.content)

## Inspeccionar salidas intermedias

La ventaja didactica del sistema multiagente es que podemos revisar que hizo cada rol antes de llegar a la respuesta final.

In [ ]:
for step in result.steps:
    print("=" * 90)
    print(f"{step.agent_name}: {step.role}")
    print("=" * 90)
    print(step.content)
    print()

## Leer la arquitectura desde codigo

El notebook es solo una interfaz de prueba. La arquitectura esta en:

```text
agents/multiagent_collaboration/
|-- agent.py      -> orquestacion, recuperacion de evidencia y flujo de roles
|-- prompts.py    -> instrucciones de Planner, Researcher, Writer y Reviewer
|-- __init__.py   -> exporta las clases principales
```

Tambien se puede ejecutar desde terminal:

```bash
python apps/sesion6_multiagent_demo.py
```

## Ejercicios

1. Cambia la pregunta de negocio y observa que evidencia recupera el sistema.
2. Modifica el prompt del `Reviewer` para que sea mas estricto.
3. Agrega un nuevo rol, por ejemplo `Risk Analyst`, antes del Writer.
4. Reemplaza la recuperacion por palabras clave por el agente RAG de sesiones 4 y 5.
5. Compara la respuesta multiagente contra una unica llamada directa al modelo.